In [20]:
# Install PySCF and GPU4PySCF (CUDA 12.x version for Colab's T4 GPU)
# cuTENSOR and CuPy are also installed for maximum tensor contraction acceleration.
!pip install pyscf gpu4pyscf-cuda12x cupy-cuda12x pyscf-dispersion geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.0/386.0 kB 7.6 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 MB 41.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 MB 14.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 12.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 46.9 MB/s eta 0:00:0000:0100:01m
  Created wheel for geometric: filename=geometric-1.1-py3-none-any.whl size=402087 sha256=2fc8ab68a5f9e04bf9a884f45d37be580c5b9d703ad9d6c6224c55645018e7b4
  Stored in directory: /root/.cache/pip/wheels/df/1e/9a/763451b0dfd8db47fc02239e33cdf138cbebdbea352bb0871b
Successfully built geometric


**Only use the next cell in case multiple GPUs are used**

In [21]:
# Must be set BEFORE any gpu4pyscf imports
import gpu4pyscf.__config__ as gpu4pyscf_config
gpu4pyscf_config.num_devices = 2

# Verify both GPUs
import cupy as cp
for i in range(2):
    with cp.cuda.Device(i):
        free, total = cp.cuda.runtime.memGetInfo()
        print(f"GPU {i}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

/usr/local/lib/python3.12/dist-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


GPU 0: 15.5 GB free / 15.6 GB total
GPU 1: 15.5 GB free / 15.6 GB total


In [22]:
import os
# Please use an xTB/other tool optimised file. Example naming is xtbopt.xyz. Please refrain from using other file formats 
# Copy the file path uploaded as a dataset in the input tab of Kaggle or system path if run locally 
xyz_filename = "/kaggle/input/datasets/pcsciprav/xtbopt-example/xtbopt.xyz"

print(f"Successfully loaded: {xyz_filename}")

Successfully loaded: /kaggle/input/datasets/pcsciprav/xtbopt-example/xtbopt.xyz


In [23]:
import cupy as cp

# Verify both GPUs are visible
n_gpus = cp.cuda.runtime.getDeviceCount()
print(f"GPUs available: {n_gpus}")
for i in range(n_gpus):
    with cp.cuda.Device(i):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(f"  GPU {i}: {props['name'].decode()}, {props['totalGlobalMem'] / 1e9:.1f} GB")

GPUs available: 2
  GPU 0: Tesla T4, 15.6 GB
  GPU 1: Tesla T4, 15.6 GB


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'  # Enable both T4s before GPU objects

import pyscf
from pyscf import gto
from gpu4pyscf.dft import rks
from pyscf.geomopt.geometric_solver import optimize
import cupy as cp

# 2. Initialize the Molecule from the uploaded xyz file
with open(xyz_filename, 'r') as f:
    lines = f.readlines()

# XYZ format: line 0 = atom count, line 1 = comment, line 2+ = atoms
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]  # skip the xtb comment line
xyz_contents = ''.join(atom_lines)

mol = gto.M(
    atom=xyz_contents,
    basis = 'def2-TZVP', # Use def2-TZVP for higher accuracy, def2-SVP is lighter and causes less OOM issues, use 3-21g for simple checks
    #ecp = 'crenbl', 
    charge=0,
    spin=0,
    verbose=4,
    unit='angstrom'
)
mol.max_memory = 10000
print(f"\nMolecule initialized: {mol.natm} atoms, {mol.nao} atomic orbitals.")

# 3. Configure the GPU-Accelerated DFT Calculation
mf_pbe = rks.RKS(mol)
mf_pbe.xc = 'pbe-d3bj'
mf_pbe.kernel()
mf = rks.RKS(mol)
mf.xc = 'r2scan-d3bj'
dm = mf_pbe.make_rdm1()
mf.kernel(dm)
mf = mf.density_fit()

# Free all VRAM before optimization
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

# Lower memory usage per SCF cycle
mf.max_memory = 10000
mf.max_cycle = 400
mf.conv_tol = 1e-8
mf.conv_tol_grad = 1e-5
mf.diis_space = 12
mf.level_shift = 0.3
mf.damp = 0.4

# DO NOT use a fresh minao guess here
# mf.init_guess = 'minao'

# 4. RUN GEOMETRY OPTIMIZATION
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

print("\nStarting GPU-accelerated Geometry Optimization...")

def save_callback(envs):
    mol_envs = envs['mol']
    mol_envs.tofile('/kaggle/working/dft_optimized_checkpoint.xyz')

mol_eq = optimize(
    mf,
    maxsteps=350,
    trust=0.03,
    tmax=0.03,
    callback=save_callback,
    assert_convergence=False
)

# 5. Save the results
output_file = 'dft_optimized.xyz'
mol_eq.tofile(output_file)

print("\n" + "="*50)
print(f"Optimization complete! New coordinates saved to: {output_file}")
print("="*50)

System: uname_result(system='Linux', node='ac9303bc783f', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Thu Apr  2 11:30:15 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 32
[INPUT] num. electrons = 240
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mol

geometric-optimize called with the following command line:
/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-b5000fb5-8237-4b96-8f65-7c1eaf07fe45.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **              ********    


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   3.063705   3.625259   0.892545    0.000000  0.000000  0.000000
   N   4.765939   0.251003   0.613203    0.000000  0.000000  0.000000
   O   4.330121   1.268830   1.066800    0.000000  0.000000  0.000000
   O   5.866695  -0.184904   0.613869    0.000000  0.000000  0.000000
   N   2.783012   3.522943   2.059665    0.000000  0.000000  0.000000
   N   1.923802   2.510104   2.606176    0.000000  0.000000  0.000000
   O   1.761632   1.504801   1.948624   -0.000000  0.000000  0.000000
   O   1.509213   2.743371   3.690510    0.000000  0.000000  0.000000
   N   1.741991   2.102478  -0.525247    0.000000  0.000000  0.000000
   N   0.330360   2.423717  -0.244417    0.000000  0.000000  0.000000
   O  -0.429565   1.612751  -0.624816    0.000000  0.000000  0.000000
   O   0.196968   3.470195   0.293427    0.000000 -0.000000  0.000000
   N   2.068679   0.859555  -0.9

Step    0 : Gradient = 3.243e-02/6.446e-02 (rms/max) Energy = -2078.3168208986
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.30000e-02 ... 1.24592e+00 1.35372e+00 1.37064e+00



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   3.052844   3.641150   0.887170   -0.010861  0.015890 -0.005375
   N   4.774512   0.241956   0.617702    0.008573 -0.009047  0.004499
   O   4.350088   1.262221   1.083115    0.019967 -0.006609  0.016315
   O   5.877093  -0.198401   0.610761    0.010398 -0.013498 -0.003108
   N   2.770361   3.560978   2.062302   -0.012651  0.038035  0.002637
   N   1.901669   2.560311   2.628863   -0.022132  0.050207  0.022687
   O   1.722126   1.536641   2.001758   -0.039506  0.031839  0.053133
   O   1.492967   2.825570   3.711204   -0.016246  0.082199  0.020694
   N   1.738005   2.101500  -0.527522   -0.003986 -0.000978 -0.002275
   N   0.324829   2.431817  -0.255836   -0.005531  0.008100 -0.011420
   O  -0.443156   1.622297  -0.633668   -0.013590  0.009546 -0.008852
   O   0.193983   3.486218   0.275322   -0.002985  0.016022 -0.018106
   N   2.070608   0.856781  -0.9

...


Geometry optimization cycle 6
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   3.050532   3.673313   0.861371    0.013000 -0.003001 -0.012598
   N   4.792198   0.255892   0.652575   -0.018557  0.026678  0.025344
   O   4.372986   1.268375   1.165440   -0.044591  0.014306  0.032993
   O   5.909484  -0.185786   0.610584   -0.009611  0.042210  0.025790
   N   2.765546   3.650166   2.065714    0.031136  0.016723 -0.003123
   N   1.845016   2.620081   2.634194    0.030185 -0.001194  0.017083
   O   1.668611   1.565726   2.048521    0.044575 -0.013138  0.033971
   O   1.417878   2.940910   3.709092    0.033517  0.015096  0.016107
   N   1.739185   2.095383  -0.519954   -0.001907 -0.007959 -0.000235
   N   0.321047   2.473539  -0.250059   -0.004232  0.005310  0.001982
   O  -0.486078   1.669562  -0.600397   -0.011905  0.007913  0.006505
   O   0.207262   3.561526   0.250935   -0.000157  0.011203 -0.004248
   N   2.080557   0.854013  -0.8

Step    5 : Displace = 3.129e-02/5.854e-02 (rms/max) Trust = 3.000e-02 (=) Grad = 8.301e-03/2.082e-02 (rms/max) E (change) = -2078.3424531736 (-2.835e-03) Quality = 0.851
Hessian Eigenvalues: 2.00441e-02 2.30000e-02 2.30000e-02 ... 1.32700e+00 1.36648e+00 2.60757e+00



Geometry optimization cycle 7
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   3.037981   3.696976   0.854452   -0.012551  0.023663 -0.006919
   N   4.823198   0.228789   0.654734    0.030999 -0.027103  0.002159
   O   4.425947   1.241796   1.181192    0.052962 -0.026580  0.015752
   O   5.934047  -0.226869   0.598534    0.024563 -0.041083 -0.012049
   N   2.744556   3.685919   2.056197   -0.020990  0.035752 -0.009517
   N   1.806303   2.654319   2.622246   -0.038713  0.034238 -0.011948
   O   1.632715   1.598243   2.041777   -0.035896  0.032517 -0.006744
   O   1.369595   2.985763   3.689655   -0.048282  0.044853 -0.019437
   N   1.746125   2.097923  -0.513442    0.006940  0.002540  0.006512
   N   0.322004   2.466375  -0.245709    0.000957 -0.007164  0.004350
   O  -0.477364   1.651940  -0.588694    0.008714 -0.017622  0.011703
   O   0.197636   3.557472   0.244765   -0.009627 -0.004053 -0.006170
   N   2.090183   0.856317  -0.8

Step    6 : Displace = 3.044e-02/6.869e-02 (rms/max) Trust = 3.000e-02 (=) Grad = 7.187e-03/1.821e-02 (rms/max) E (change) = -2078.3437012400 (-1.248e-03) Quality = 0.810
Hessian Eigenvalues: 2.21762e-02 2.30000e-02 2.30000e-02 ... 1.35895e+00 1.42468e+00 2.51752e+00



Geometry optimization cycle 8
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   3.045064   3.694472   0.846782    0.007083 -0.002505 -0.007671
   N   4.818009   0.237587   0.667455   -0.005189  0.008798  0.012721
   O   4.416092   1.240717   1.205973   -0.009856 -0.001079  0.024780
   O   5.931824  -0.207334   0.597824   -0.002223  0.019535 -0.000710
   N   2.744177   3.703647   2.044909   -0.000379  0.017728 -0.011288
   N   1.770974   2.680642   2.615539   -0.035329  0.026323 -0.006707
   O   1.593206   1.620166   2.048729   -0.039509  0.021922  0.006952
   O   1.325860   3.036407   3.670296   -0.043735  0.050644 -0.019359
   N   1.749862   2.096140  -0.511130    0.003737 -0.001783  0.002312
   N   0.324848   2.485441  -0.249179    0.002844  0.019066 -0.003470
   O  -0.486616   1.680032  -0.582844   -0.009253  0.028092  0.005850
   O   0.215592   3.583750   0.226479    0.017956  0.026277 -0.018287
   N   2.088388   0.852246  -0.8

In [ ]:
import os
for f in os.listdir('/kaggle/working'):
    print(f)
import glob
print(glob.glob('/kaggle/working/*.xyz'))
print(glob.glob('/kaggle/working/**/*.xyz', recursive=True))

In [ ]:
# Cell 4: Vibrational Frequencies and Thermochemistry
from gpu4pyscf.dft import rks
from gpu4pyscf.hessian import rks as gpu_hessian
from pyscf.hessian import thermo
import cupy as cp
import numpy as np
from pyscf import gto
import numpy as np

with open('/kaggle/working/dft_optimized_checkpoint.xyz', 'r') as f:
    lines = f.readlines()
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]
xyz_contents = ''.join(atom_lines)

mol_eq = gto.M(
    atom = xyz_contents,
    basis = 'def2-TZVP',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol_eq.max_memory = 16000

print("1. Setting up new SCF for the optimized geometry...")
mf_eq = rks.RKS(mol_eq)
mf_eq.xc = 'r2scan-d3bj'
mf_eq = mf_eq.density_fit()
mf_eq.conv_tol = 1e-9
e_opt = mf_eq.kernel()

print("\n2. Calculating analytical Hessian (Frequencies) on the GPU...")
hess = gpu_hessian.Hessian(mf_eq)
hess_matrix = hess.kernel()
# === AUTO-FIX IMAGINARY MODES LOOP ===

max_iter = 5
step = 0.1  # distortion magnitude (Å)

for i in range(max_iter):
    print("\n" + "="*40)
    print(f"Iteration {i+1}")
    print("="*40)

    # --- Compute Hessian ---
    hess = gpu_hessian.Hessian(mf_eq)
    hess_matrix = hess.kernel()

    # --- Reshape to 3N x 3N ---
    hess_2d = hess_matrix.reshape(3*n_atoms, 3*n_atoms)

    # --- Eigen decomposition ---
    eigvals, eigvecs = np.linalg.eigh(hess_2d)

    # --- Find REAL negative modes (ignore tiny noise) ---
    neg_indices = np.where(eigvals < -1e-3)[0]

    print(f"Negative eigenvalues: {len(neg_indices)}")
    print("Smallest eigenvalues:", eigvals[:6])

    # --- Stop if clean ---
    if len(neg_indices) == 0:
        print("✅ Reached true minimum (no real imaginary modes)")
        break

    # --- Pick most negative mode ---
    mode_idx = neg_indices[0]
    mode = eigvecs[:, mode_idx].reshape(n_atoms, 3)

    coords = mol_eq.atom_coords()

    # --- Try BOTH directions ---
    best_energy = None
    best_coords = None

    for direction in [+1, -1]:
        trial_coords = coords + direction * step * mode

        mol_trial = mol_eq.copy()
        mol_trial.set_geom_(trial_coords, unit='angstrom')

        mf_trial = rks.RKS(mol_trial)
        mf_trial.xc = 'r2scan-d3bj'
        mf_trial = mf_trial.density_fit()
        mf_trial.conv_tol = 1e-9

        e_trial = mf_trial.kernel()

        print(f"  Direction {direction:+} Energy: {e_trial:.6f}")

        if best_energy is None or e_trial < best_energy:
            best_energy = e_trial
            best_coords = trial_coords

    # --- Update geometry to best ---
    mol_eq.set_geom_(best_coords, unit='angstrom')

    # --- Re-optimise ---
    print("Re-optimising...")
    mf_eq = rks.RKS(mol_eq)
    mf_eq.xc = 'r2scan-d3bj'
    mf_eq = mf_eq.density_fit()
    mf_eq.conv_tol = 1e-9
    mf_eq.kernel()

print("\nLoop complete.")
print("\n3. Calculating Thermochemistry (298.15 K, 1 atm)...")
freq_info = thermo.harmonic_analysis(mol_eq, hess_matrix)
thermo_data = thermo.thermo(mf_eq, freq_info['freq_au'], 298.15, 101325.0)

print("\n--- Thermochemistry Summary ---")
thermo.dump_thermo(mol_eq, thermo_data)

# MEMORY MANAGEMENT & BACKUP
print("\n4. Saving data to disk and purging VRAM...")

# Step A: Save the massive Hessian matrix to a file so we don't lose it
np.save('hessian_matrix.npy', np.asarray(hess_matrix))
print("   -> Hessian securely saved to 'hessian_matrix.npy'")

# Step B: Explicitly delete the giant objects from Python's memory
del hess
del freq_info
del thermo_data

# Step C: Force the GPU to empty the trash now that Python let go of the variables
cp.get_default_memory_pool().free_all_blocks()
print("   -> GPU VRAM successfully flushed!")

print("\n" + "="*50)
print("FREQUENCY & THERMO CALCULATION COMPLETE")
print("="*50)

In [ ]:
import numpy as np
import cupy as cp

print("="*50)
print("HESSIAN / EIGENVALUE SUMMARY")
print("="*50)

H = hess_matrix
if isinstance(H, cp.ndarray):
    H = cp.asnumpy(H)

# block Hessian (natm,natm,3,3) -> full matrix
natm = H.shape[0]
Hfull = H.transpose(0,2,1,3).reshape(3*natm, 3*natm)

w = np.linalg.eigvalsh(Hfull)

print("Shape of full Hessian:", Hfull.shape)
print("Smallest 20 eigenvalues:")
print(w[:20])
print("Largest 10 eigenvalues:")
print(w[-10:])

tol = 1e-8
nneg = np.sum(w < -tol)
nzero = np.sum(np.abs(w) <= tol)
npos = np.sum(w > tol)

print("Negative eigenvalues:", nneg)
print("Near-zero eigenvalues:", nzero)
print("Positive eigenvalues:", npos)
print("="*50)
print("SOFTEST MODE ATOMIC PARTICIPATION")
print("="*50)

eigvals, eigvecs = np.linalg.eigh(Hfull)

for mode in range(min(5, len(eigvals))):
    vec = eigvecs[:, mode].reshape(natm, 3)
    atom_amp = np.linalg.norm(vec, axis=1)
    top = np.argsort(atom_amp)[::-1][:10]
    print(f"\nMode {mode}  eigenvalue = {eigvals[mode]: .8f}")
    for i in top:
        print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  amplitude = {atom_amp[i]:.6f}")
del hess_matrix

In [ ]:
# Cell 5: Excited States (TD-DFT) for UV-Vis
from gpu4pyscf.tdscf import rks as gpu_tdscf
import cupy as cp
from pyscf import gto
from gpu4pyscf.dft import rks

with open('dft_optimized_checkpoint.xyz', 'r') as f:
    lines = f.readlines()
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]
xyz_contents = ''.join(atom_lines)

mol_eq = gto.M(
    atom = xyz_contents,
    basis = 'def2-TZVP',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol_eq.max_memory = 10000

mf_eq = rks.RKS(mol_eq)
mf_eq.xc = 'r2scan-d3bj'
mf_eq = mf_eq.density_fit()
mf_eq.conv_tol = 1e-9
mf_eq.kernel()
print("Cleaning up GPU VRAM from previous calculations...")
# ==========================================
# CRITICAL VRAM CLEANUP
# Forces the GPU to release cached memory from the Hessian calc
# ==========================================
cp.get_default_memory_pool().free_all_blocks()

print("\nStarting GPU-accelerated TD-DFT...")
print("Calculating the first 5 singlet excited states...\n")

# Initialize Time-Dependent DFT using our optimized molecule's SCF object (mf_eq)
td = gpu_tdscf.TDDFT(mf_eq)
td.nstates = 5       # Number of excited states to calculate
td.singlet = True    # Look for singlet-singlet transitions

# ==========================================
# VRAM OPTIMIZATION FOR TD-DFT
# Restricts the Davidson solver's maximum memory footprint
# ==========================================
td.max_space = 15

# Run the calculation
td.kernel()

print("\n" + "="*50)
print("TD-DFT EXCITATION ENERGIES:")
print("="*50)
# This will print the transition energies (in eV), wavelengths (in nm), and oscillator strengths (intensity)
td.analyze()

In [ ]:
print("="*50)
print("TDDFT SUMMARY")
print("="*50)

print("Excitation energies (eV):")
print(cp.asnumpy(td.e) * 27.211386245988 if hasattr(td.e, "get") or "cupy" in str(type(td.e)).lower() else td.e * 27.211386245988)

print("\nOscillator strengths:")
try:
    print(td.oscillator_strength())
except Exception as e:
    print("Could not get oscillator strengths directly:", e)

In [ ]:
import cupy as cp
from pyscf.scf import hf

print("="*50)
print("1. MOLECULAR DIPOLE MOMENT")
print("="*50)
dipole = mf_eq.dip_moment()

print("\n" + "="*50)
print("2. MULLIKEN ATOMIC CHARGES")
print("="*50)

dm = mf_eq.make_rdm1()
s = mol_eq.intor('int1e_ovlp')

# Convert GPU array to CPU NumPy array
dm = cp.asnumpy(dm)

mulliken_pop, mulliken_chg = hf.mulliken_pop(mol_eq, dm, s=s)

print("\nAtom | Charge")
print("-" * 20)
for i, charge in enumerate(mulliken_chg):
    atom_symbol = mol_eq.atom_symbol(i)
    print(f"{i:2d} {atom_symbol:2s} | {charge: .4f}")

In [ ]:
print("="*50)
print("ATOM INDEX MAP")
print("="*50)

for i in range(mol_eq.natm):
    x, y, z = mol_eq.atom_coord(i)
    print(f"{i:2d} {mol_eq.atom_symbol(i):2s}  {x: .6f} {y: .6f} {z: .6f}")

In [ ]:
import numpy as np

print("="*50)
print("CHARGE SUMMARY")
print("="*50)

charges = np.array(mulliken_chg)

print("Min charge:", charges.min())
print("Max charge:", charges.max())
print("Mean charge:", charges.mean())
print("Std dev:", charges.std())

print("\nTop 10 most positive atoms")
for i in np.argsort(charges)[::-1][:10]:
    print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  q = {charges[i]: .6f}")

print("\nTop 10 most negative atoms")
for i in np.argsort(charges)[:10]:
    print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  q = {charges[i]: .6f}")

In [ ]:
# Cell 7: Save the optimized coordinates
output_xyz = 'dft_optimized_final.xyz'
mol_eq.tofile(output_xyz)
print(f"Saved to /kaggle/working/{output_xyz}")
print("Download it from the Output tab on the right panel.")

In [2]:
import os

base = "/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3"

os.environ["PATH"] = base + "/compilers/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = base + "/compilers/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
!wget https://developer.download.nvidia.com/hpc-sdk/24.3/nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz
!tar -xvf nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz
%cd nvhpc_2024_243_Linux_x86_64_cuda_12.3
!yes "" | ./install
!nvfortran --version
print("Done")

--2026-04-02 11:05:12--  https://developer.download.nvidia.com/hpc-sdk/24.3/nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.1.33.7, 23.1.33.10
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.1.33.7|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5769368135 (5.4G) [application/x-gzip]
Saving to: ‘nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz’

nvhpc_2024_243_Linu 100%[===================>]   5.37G  17.0MB/s    in 4m 29s  

2026-04-02 11:09:41 (20.4 MB/s) - ‘nvhpc_2024_243_Linux_x86_64_cuda_12.3.tar.gz’ saved [5769368135/5769368135]

nvhpc_2024_243_Linux_x86_64_cuda_12.3/install
nvhpc_2024_243_Linux_x86_64_cuda_12.3/install_components/
nvhpc_2024_243_Linux_x86_64_cuda_12.3/install_components/NVHPCConfig.cmake
nvhpc_2024_243_Linux_x86_64_cuda_12.3/install_components/NVHPCConfigVersion.cmake
nvhpc_2024_243_Linux_x86_64_cuda_12.3/install_components/install
nvhpc

In [19]:
!make pw -j2

test -d bin || mkdir bin
( cd UtilXlib ; make TLDEPS= all || exit 1 )
cd install ; make -f extlibs_makefile libcuda
make[1]: Entering directory '/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2/UtilXlib'
nvfortran -fast -Mcache_align -Mpreprocess -Mlarge_arrays -mp -D__PGI -D__CUDA -D__DFTI  -cuda -gpu=cc75,cuda12.3 -I/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2//external/devxlib/src -I/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2//external/devxlib/include -acc -I/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2//external/devxlib/src -I. -I/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2//include -I/opt/intel/mkl/include -I. -c parallel_include.f90
make[1]: Entering directory '/kaggle/working/nvhpc_2024_243_Linux_x86_64_cuda_12.3/q-e-qe-7.2/install'
initializing external/devxlib submodule ...
hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. 